# Clase 072 — Curvas de aprendizaje y bias-variance

Diagnosticamos **alto sesgo** vs **alta varianza** leyendo curvas de aprendizaje (`learning_curve`), encontramos el grado óptimo con `validation_curve` y medimos empíricamente la descomposición $bias^2 + var$ vía bootstrap.

Requiere: `numpy`, `scikit-learn`, `matplotlib`.

## 1. Dataset con ruido conocido

$y = 0.5x^2 + x + 2 + \varepsilon$, con $\varepsilon\sim\mathcal{N}(0, \sigma^2)$ y $\sigma=1$. Conocer $\sigma$ nos permite comparar el error irreducible con la varianza medida.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
m = 200
X = 6 * np.random.rand(m, 1) - 3
sigma = 1.0
y = 0.5 * X[:, 0]**2 + X[:, 0] + 2 + np.random.randn(m) * sigma
print('dataset:', X.shape, '| ruido σ =', sigma, '| σ² =', sigma**2)

## 2. Curva de aprendizaje: underfitting

Una regresión **lineal** sobre datos cuadráticos subajusta: las curvas de train y validación convergen **altas y juntas**. Más datos no ayudan.

In [ ]:
from sklearn.model_selection import learning_curve
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures

def plot_lc(model, X, y, title, ax):
    sizes, tr, va = learning_curve(
        model, X, y, cv=5, scoring='neg_root_mean_squared_error',
        train_sizes=np.linspace(0.1, 1.0, 8))
    ax.plot(sizes, -tr.mean(1), 'o-', label='train')
    ax.plot(sizes, -va.mean(1), 's-', label='validación')
    ax.set_title(title); ax.set_xlabel('train size'); ax.set_ylabel('RMSE'); ax.legend()
    return -tr.mean(1)[-1], -va.mean(1)[-1]

fig, ax = plt.subplots(figsize=(7, 4))
tr_f, va_f = plot_lc(LinearRegression(), X, y, 'Regresión lineal: underfitting', ax)
plt.tight_layout(); plt.show()
print(f'RMSE final -> train {tr_f:.3f} | val {va_f:.3f} (curvas altas y juntas = alto sesgo)')

## 3. Underfit vs fit vs overfit

Grado 2 ajusta bien (curvas bajas y juntas); grado 10 muestra un **gap persistente** entre train y validación (alta varianza).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plot_lc(make_pipeline(PolynomialFeatures(2), LinearRegression()), X, y, 'Grado 2: buen ajuste', axes[0])
tr10, va10 = plot_lc(make_pipeline(PolynomialFeatures(10), LinearRegression()), X, y,
                     'Grado 10: gap = overfitting', axes[1])
plt.tight_layout(); plt.show()
print('gap final grado 10 (val - train):', round(va10 - tr10, 3))

## 4. Validation curve: sweet spot

`validation_curve` varía un **hiperparámetro** (el grado), no el tamaño de train. El mínimo de la RMSE de validación marca la capacidad óptima.

In [ ]:
from sklearn.model_selection import validation_curve

degrees = np.arange(1, 16)
model = make_pipeline(PolynomialFeatures(1), LinearRegression())
tr, va = validation_curve(
    model, X, y, param_name='polynomialfeatures__degree', param_range=degrees,
    cv=5, scoring='neg_root_mean_squared_error')
val_rmse = -va.mean(1)
best_d = degrees[int(np.argmin(val_rmse))]
print('sweet spot (grado con menor RMSE de validación):', best_d)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(degrees, -tr.mean(1), 'o-', label='train')
ax.plot(degrees, val_rmse, 's-', label='validación')
ax.axvline(best_d, color='k', ls=':'); ax.legend()
ax.set_xlabel('grado'); ax.set_ylabel('RMSE'); ax.set_title('Validation curve: barrido del grado')
plt.tight_layout(); plt.show()
assert 2 <= best_d <= 6

## 5. Bias-variance empírico (bootstrap)

Entrenamos 100 modelos de grado 10 sobre remuestreos del dataset y, en una grilla de test, medimos el $bias^2$ (contra la función verdadera) y la **varianza** de las predicciones. El grado 10 debe tener varianza dominante.

In [ ]:
rng = np.random.default_rng(0)
x_test = np.linspace(-3, 3, 50).reshape(-1, 1)
f_true = 0.5 * x_test[:, 0]**2 + x_test[:, 0] + 2
n_boot = 100
preds = np.zeros((n_boot, len(x_test)))
for b in range(n_boot):
    idx = rng.integers(0, m, m)
    model = make_pipeline(PolynomialFeatures(10), LinearRegression()).fit(X[idx], y[idx])
    preds[b] = model.predict(x_test)

mean_pred = preds.mean(0)
bias2 = np.mean((mean_pred - f_true)**2)
var = np.mean(preds.var(0))
print(f'bias² = {bias2:.4f} | varianza = {var:.4f} | suma = {bias2 + var:.4f} | σ² = {sigma**2:.4f}')
assert var > bias2
print('OK: el grado 10 (alta capacidad) está dominado por la varianza, no por el sesgo')

## Ejercicios

1. **¿Más datos ayudan?** Para el polinomio de grado 10, extendé el dataset a 2000 puntos y volvé a graficar la curva de aprendizaje. Cuantificá cuánto se cierra el gap.
2. **Árboles.** Repetí las curvas de aprendizaje con `DecisionTreeRegressor(max_depth=d)` para $d\in\{2, 5, 10, \text{None}\}$ y clasificá cada uno como underfit / fit / overfit.
3. **Clasificación.** Adaptá `plot_lc` a un problema de clasificación con `scoring='neg_log_loss'` y verificá que la interpretación del gap es análoga.
4. **Piso de error.** Comprobá que la RMSE de validación de un buen modelo nunca baja de $\sigma$: apuntar a cero es imposible con ruido irreducible.

## Conclusiones

- La **curva de aprendizaje** varía el tamaño de train; la **validation curve** varía un hiperparámetro: no son intercambiables.
- Curvas altas y juntas ⇒ **alto sesgo** (más datos no ayudan, hay que subir capacidad); gap persistente ⇒ **alta varianza** (más datos o regularización).
- El error de generalización se descompone en $bias^2 + var + \sigma^2$; el $\sigma^2$ es el piso irreducible.
- Lo que se lee para diagnosticar no es el error absoluto de un punto, sino el **gap** y la **tendencia asintótica**.

## ✅ Soluciones de los ejercicios

Soluciones trabajadas y comentadas de los ejercicios de la seccion 🧪 **Ejercicios** del README. Cada bloque es autocontenido, se ejecuta **sin internet** y en pocos segundos. Intenta resolver cada ejercicio por tu cuenta antes de mirar la solucion.

### Ejercicio 1 — Curva de aprendizaje de una regresion lineal
Curvas altas y juntas = alto **bias** (underfit).

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import learning_curve

def plot_lc(estimator, X, y, title, ax):
    sizes, tr, va = learning_curve(estimator, X, y, cv=5,
        train_sizes=np.linspace(0.1, 1.0, 8), scoring='neg_root_mean_squared_error')
    ax.plot(sizes, -tr.mean(1), 'o-', label='train'); ax.plot(sizes, -va.mean(1), 's-', label='val')
    ax.set_title(title); ax.set_xlabel('m'); ax.set_ylabel('RMSE'); ax.legend()

fig, ax = plt.subplots(figsize=(6, 4))
plot_lc(LinearRegression(), X, y, 'lineal: underfit', ax)
plt.tight_layout(); plt.show()

### Ejercicio 2 — Aumentar capacidad
Grado 2 = buen fit; grado 10 = overfit (gap train-val).

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plot_lc(make_pipeline(PolynomialFeatures(2), LinearRegression()), X, y, 'grado 2: fit', axes[0])
plot_lc(make_pipeline(PolynomialFeatures(10), LinearRegression()), X, y, 'grado 10: overfit', axes[1])
plt.tight_layout(); plt.show()

### Ejercicio 3 — ¿Mas datos ayudan?
Con 2000 puntos el gap del grado 10 se cierra (baja la varianza).

In [ ]:
rng = np.random.RandomState(1)
Xbig = 6 * rng.rand(2000, 1) - 3
ybig = 0.5 * Xbig[:, 0] ** 2 + Xbig[:, 0] + 2 + rng.randn(2000) * sigma
fig, ax = plt.subplots(figsize=(6, 4))
plot_lc(make_pipeline(PolynomialFeatures(10), LinearRegression()), Xbig, ybig, 'grado 10 con 2000 puntos', ax)
plt.tight_layout(); plt.show()
print('Mas datos -> el gap train-val se cierra: la varianza baja.')

### Ejercicio 4 — Validation curve sobre el grado
Buscamos el sweet spot donde el val RMSE es minimo.

In [ ]:
from sklearn.model_selection import validation_curve
degs = np.arange(1, 16)
tr, va = validation_curve(make_pipeline(PolynomialFeatures(), LinearRegression()),
    X, y, param_name='polynomialfeatures__degree', param_range=degs, cv=5,
    scoring='neg_root_mean_squared_error')
plt.figure(figsize=(7, 4)); plt.plot(degs, -tr.mean(1), 'o-', label='train'); plt.plot(degs, -va.mean(1), 's-', label='val')
plt.legend(); plt.xlabel('grado'); plt.ylabel('RMSE'); plt.title('sweet spot')
plt.tight_layout(); plt.show()
print('mejor grado:', degs[int(np.argmin(-va.mean(1)))])

### Ejercicio 5 — Bias-variance empirico
Con 100 bootstraps de un polinomio grado 10, medimos bias² y varianza.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
x_grid = np.linspace(-3, 3, 50).reshape(-1, 1)
f_true = 0.5 * x_grid[:, 0] ** 2 + x_grid[:, 0] + 2
rng = np.random.RandomState(0); preds = []
for _ in range(100):
    idx = rng.randint(0, len(X), len(X))
    mdl = make_pipeline(PolynomialFeatures(10), LinearRegression()).fit(X[idx], y[idx])
    preds.append(mdl.predict(x_grid))
preds = np.array(preds)
bias2 = ((preds.mean(0) - f_true) ** 2).mean()
var = preds.var(0).mean()
print(f'bias^2={bias2:.3f}  var={var:.3f}  bias^2+var={bias2 + var:.3f}  (sigma^2={sigma ** 2})')
print('Grado 10 -> la varianza domina: ese es el costo de la alta capacidad.')